# Notebook 05 — Práctica

Quinto y último sub-bloque del Tema 06. **12 ejercicios graduales** que combinan bloques anónimos, control de flujo, funciones y procedimientos sobre Northwind DWH.

Cada ejercicio sigue el patrón ya conocido:

1. **Enunciado** en markdown.
2. **Celda vacía** para tu solución.
3. **Solución** colapsable (`<details>`) — ábrela solo cuando hayas intentado.

Tres niveles:

- **Fácil (1-4):** bloques anónimos con `IF`, variables, `RAISE NOTICE`.
- **Medio (5-8):** funciones con parámetros, `FOR ... IN SELECT`, retorno escalar y `TABLE`.
- **Difícil (9-12):** procedimientos, manejo de excepciones, decisión entre función/procedimiento/SQL puro.

## Setup

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel y vuelve a correr.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine

## Nivel fácil (1-4)

### Ejercicio 1 — Saludo según hora del día

Crea un bloque anónimo que declare una variable `hora` con el valor de `EXTRACT(hour FROM CURRENT_TIMESTAMP)` y muestre con `RAISE NOTICE`:

- `'Buenos días'` si la hora es menor a 12
- `'Buenas tardes'` si está entre 12 y 18 (inclusive)
- `'Buenas noches'` si es mayor a 18

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
DO $$
DECLARE
    hora INTEGER := EXTRACT(hour FROM CURRENT_TIMESTAMP);
BEGIN
    IF hora < 12 THEN
        RAISE NOTICE 'Buenos días (hora=%)', hora;
    ELSIF hora <= 18 THEN
        RAISE NOTICE 'Buenas tardes (hora=%)', hora;
    ELSE
        RAISE NOTICE 'Buenas noches (hora=%)', hora;
    END IF;
END
$$;
```
</details>

### Ejercicio 2 — Sumatoria del 1 al N

Bloque anónimo que calcule la suma del 1 al 100 usando un `FOR` loop y la muestre con `RAISE NOTICE`. Verifica que el resultado sea 5050.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
DO $$
DECLARE
    suma INTEGER := 0;
BEGIN
    FOR i IN 1..100 LOOP
        suma := suma + i;
    END LOOP;
    RAISE NOTICE 'Suma 1 a 100 = %', suma;
END
$$;
```

Nota: esto se podría hacer en SQL puro con `SELECT SUM(generate_series(1, 100));` — el ejercicio es para practicar el `FOR` loop.
</details>

### Ejercicio 3 — Categorizar conteo de clientes

Bloque anónimo que cuente clientes en `dim_customer`, y según el conteo imprima:

- `'Base pequeña (<50)'`
- `'Base mediana (50-200)'`
- `'Base grande (>200)'`

Usa `SELECT ... INTO` para asignar el conteo a la variable.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
DO $$
DECLARE
    total INTEGER;
BEGIN
    SELECT COUNT(*) INTO total FROM northwind_dwh.dim_customer;
    
    IF total < 50 THEN
        RAISE NOTICE 'Base pequeña (<50): % clientes', total;
    ELSIF total <= 200 THEN
        RAISE NOTICE 'Base mediana (50-200): % clientes', total;
    ELSE
        RAISE NOTICE 'Base grande (>200): % clientes', total;
    END IF;
END
$$;
```
</details>

### Ejercicio 4 — Listar productos por categoría

Bloque anónimo que itere sobre las categorías de `dim_product` y para cada una imprima:

```
Categoría X: N productos
```

Usa `FOR ... IN SELECT`.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
DO $$
DECLARE
    fila RECORD;
BEGIN
    FOR fila IN
        SELECT category_name, COUNT(*) AS productos
          FROM northwind_dwh.dim_product
         GROUP BY category_name
         ORDER BY productos DESC
    LOOP
        RAISE NOTICE 'Categoría %: % productos', fila.category_name, fila.productos;
    END LOOP;
END
$$;
```
</details>

## Nivel medio (5-8)

### Ejercicio 5 — Función `aplicar_iva`

Crea una función `aplicar_iva(precio NUMERIC, tasa NUMERIC DEFAULT 16)` que devuelva el precio más el IVA (porcentaje). Marca la función como `IMMUTABLE` y `LANGUAGE sql` (cabe en una sola query).

Pruébala con `SELECT aplicar_iva(100);` y `SELECT aplicar_iva(100, 8);`.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
CREATE OR REPLACE FUNCTION aplicar_iva(
    precio NUMERIC,
    tasa   NUMERIC DEFAULT 16
) RETURNS NUMERIC
AS 'SELECT ROUND(precio * (1 + tasa / 100), 2)'
LANGUAGE sql IMMUTABLE;

SELECT aplicar_iva(100) AS con_16, aplicar_iva(100, 8) AS con_8;
```

`LANGUAGE sql IMMUTABLE` permite al optimizador *inlinear* la función directamente en queries que la usen — más rápido que `LANGUAGE plpgsql`.
</details>

### Ejercicio 6 — Función `dias_envio`

Crea una función `dias_envio(order_id INTEGER)` que devuelva el número de días entre `order_date` y `shipped_date` para un pedido específico de Northwind. Si el pedido no se ha enviado, devuelve `NULL`.

Pista: usa `northwind_oltp.orders`. La firma debe ser `RETURNS INTEGER`.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
CREATE OR REPLACE FUNCTION dias_envio(p_order_id INTEGER) RETURNS INTEGER AS $$
DECLARE
    fecha_orden DATE;
    fecha_envio DATE;
BEGIN
    SELECT order_date, shipped_date
      INTO fecha_orden, fecha_envio
      FROM northwind_oltp.orders
     WHERE order_id = p_order_id;
    
    IF NOT FOUND THEN
        RAISE EXCEPTION 'Pedido % no existe', p_order_id;
    END IF;
    
    IF fecha_envio IS NULL THEN
        RETURN NULL;
    END IF;
    
    RETURN fecha_envio - fecha_orden;
END;
$$ LANGUAGE plpgsql STABLE;

SELECT dias_envio(10248) AS dias;
```
</details>

### Ejercicio 7 — Función `top_n_productos`

Crea una función `top_n_productos(n INTEGER)` que devuelva las top N categorías por ventas netas (`SUM(line_total)`), con columnas `categoria`, `ventas_netas`, `lineas`.

Pista: usa `RETURNS TABLE(...)` y `RETURN QUERY`.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
CREATE OR REPLACE FUNCTION top_n_productos(p_n INTEGER)
RETURNS TABLE(
    categoria    TEXT,
    ventas_netas NUMERIC,
    lineas       BIGINT
) AS $$
BEGIN
    RETURN QUERY
    SELECT dp.category_name::TEXT,
           ROUND(SUM(fs.line_total), 2),
           COUNT(*)::BIGINT
      FROM northwind_dwh.fact_sales fs
      JOIN northwind_dwh.dim_product dp USING (product_key)
     GROUP BY dp.category_name
     ORDER BY 2 DESC
     LIMIT p_n;
END;
$$ LANGUAGE plpgsql STABLE;

SELECT * FROM top_n_productos(3);
```
</details>

### Ejercicio 8 — Función con excepción manejada

Crea una función `tasa_de_envio_segura(p_order_id INTEGER)` que use `dias_envio(p_order_id)` (del Ej. 6) y devuelva:

- `'rápido'` si los días son ≤ 3
- `'normal'` si están entre 4 y 7
- `'lento'` si son > 7
- `'pendiente'` si `dias_envio` devuelve NULL
- `'desconocido'` si `dias_envio` lanza excepción (pedido no existe)

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
CREATE OR REPLACE FUNCTION tasa_de_envio_segura(p_order_id INTEGER) RETURNS TEXT AS $$
DECLARE
    d INTEGER;
BEGIN
    BEGIN
        d := dias_envio(p_order_id);
    EXCEPTION
        WHEN OTHERS THEN
            RETURN 'desconocido';
    END;
    
    IF d IS NULL THEN
        RETURN 'pendiente';
    ELSIF d <= 3 THEN
        RETURN 'rápido';
    ELSIF d <= 7 THEN
        RETURN 'normal';
    ELSE
        RETURN 'lento';
    END IF;
END;
$$ LANGUAGE plpgsql STABLE;

SELECT tasa_de_envio_segura(10248) AS pedido_real,
       tasa_de_envio_segura(99999) AS pedido_inexistente;
```

El sub-bloque `BEGIN ... EXCEPTION ... END;` captura cualquier excepción de `dias_envio` y devuelve `'desconocido'` sin abortar la función.
</details>

## Nivel difícil (9-12)

### Ejercicio 9 — Procedimiento `reportar_top_clientes`

Crea un **procedimiento** que reciba un parámetro `p_n INTEGER` y un parámetro `OUT p_total NUMERIC`. El procedimiento debe:

1. Iterar sobre los top N clientes por ventas netas.
2. Para cada uno, hacer `RAISE NOTICE` con: nombre del cliente y total de ventas.
3. Acumular el gran total en `p_total`.

Invócalo con `CALL reportar_top_clientes(5, NULL);`.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
CREATE OR REPLACE PROCEDURE reportar_top_clientes(
    p_n     INTEGER,
    OUT p_total NUMERIC
) AS $$
DECLARE
    fila RECORD;
BEGIN
    p_total := 0;
    
    FOR fila IN
        SELECT dc.company_name, SUM(fs.line_total) AS total
          FROM northwind_dwh.fact_sales fs
          JOIN northwind_dwh.dim_customer dc USING (customer_key)
         GROUP BY dc.company_name
         ORDER BY 2 DESC
         LIMIT p_n
    LOOP
        RAISE NOTICE '% — $%', fila.company_name, ROUND(fila.total, 2);
        p_total := p_total + fila.total;
    END LOOP;
    
    RAISE NOTICE 'Gran total top %: $%', p_n, ROUND(p_total, 2);
END;
$$ LANGUAGE plpgsql;

CALL reportar_top_clientes(5, NULL);
```
</details>

### Ejercicio 10 — Procedimiento de validación de DWH

Crea un procedimiento `validar_dwh()` que verifique tres invariantes y haga `RAISE NOTICE`/`WARNING` según corresponda:

1. Conteo de `fact_sales` coincide con conteo de `northwind_oltp.order_details`.
2. Ninguna fila de `fact_sales` tiene `customer_key`, `product_key` o `employee_key` NULL.
3. `SUM(line_total)` en DWH es ≈ `SUM(quantity * unit_price * (1 - discount))` en OLTP (tolerancia ±$10 por la corrección REAL→NUMERIC).

Si alguna falla, `RAISE WARNING`. Si todas pasan, `RAISE NOTICE 'DWH consistente.'`.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
CREATE OR REPLACE PROCEDURE validar_dwh() AS $$
DECLARE
    c_fact   INTEGER;
    c_oltp   INTEGER;
    c_nulls  INTEGER;
    s_dwh    NUMERIC;
    s_oltp   NUMERIC;
    todo_ok  BOOLEAN := TRUE;
BEGIN
    -- 1. Conteo
    SELECT COUNT(*) INTO c_fact FROM northwind_dwh.fact_sales;
    SELECT COUNT(*) INTO c_oltp FROM northwind_oltp.order_details;
    IF c_fact <> c_oltp THEN
        RAISE WARNING 'Conteos no coinciden: fact=%, oltp=%', c_fact, c_oltp;
        todo_ok := FALSE;
    END IF;
    
    -- 2. FKs obligatorias
    SELECT COUNT(*) INTO c_nulls
      FROM northwind_dwh.fact_sales
     WHERE customer_key IS NULL OR product_key IS NULL OR employee_key IS NULL;
    IF c_nulls > 0 THEN
        RAISE WARNING '% filas con FK NULL en fact_sales', c_nulls;
        todo_ok := FALSE;
    END IF;
    
    -- 3. SUM consistente
    SELECT SUM(line_total) INTO s_dwh FROM northwind_dwh.fact_sales;
    SELECT SUM(quantity * unit_price * (1 - discount)) INTO s_oltp
      FROM northwind_oltp.order_details;
    IF ABS(s_dwh - s_oltp) > 10 THEN
        RAISE WARNING 'SUM divergente: DWH=%, OLTP=%, delta=%',
            ROUND(s_dwh, 2), ROUND(s_oltp, 2), ROUND(s_dwh - s_oltp, 2);
        todo_ok := FALSE;
    END IF;
    
    IF todo_ok THEN
        RAISE NOTICE 'DWH consistente: %  filas, sum=$%', c_fact, ROUND(s_dwh, 2);
    END IF;
END;
$$ LANGUAGE plpgsql;

CALL validar_dwh();
```
</details>

### Ejercicio 11 — "¿Función o procedimiento?"

Para cada uno de los siguientes casos, decide si conviene **función**, **procedimiento**, o **SQL puro**, y explica brevemente por qué:

1. Calcular el margen porcentual de un precio dado el costo (`(precio - costo) / precio * 100`).
2. Refrescar nightly una tabla `aggregates_diarios` desde `fact_sales` (vaciar y recargar).
3. Obtener el ranking de top 10 vendedores con cantidad de pedidos.
4. Calcular ventas totales agrupadas por mes para 1997.

Escribe tu razonamiento en una celda markdown debajo, y opcionalmente implementa una de las opciones.

<details>
<summary><strong>Solución</strong></summary>

1. **Función `LANGUAGE sql IMMUTABLE`** — cómputo determinista, reutilizable en queries, sin efectos secundarios:
   ```sql
   CREATE FUNCTION margen_pct(precio NUMERIC, costo NUMERIC) RETURNS NUMERIC
   AS 'SELECT ROUND((precio - costo) / NULLIF(precio, 0) * 100, 2)'
   LANGUAGE sql IMMUTABLE;
   ```

2. **Procedimiento** — efecto secundario (modifica tabla), necesita posiblemente control de transacciones para commit por chunks:
   ```sql
   CREATE PROCEDURE refrescar_aggregates_diarios() AS $$
   BEGIN
       TRUNCATE TABLE aggregates_diarios;
       INSERT INTO aggregates_diarios SELECT ... FROM fact_sales ...;
   END;
   $$ LANGUAGE plpgsql;
   ```

3. **Función `RETURNS TABLE`** — devuelve un set de filas reutilizable, sin efectos secundarios:
   ```sql
   CREATE FUNCTION top_vendedores(n INTEGER) RETURNS TABLE(empleado TEXT, pedidos INTEGER) AS $$ ... $$;
   ```

4. **SQL puro** — no requiere lógica procedural, es una query analítica:
   ```sql
   SELECT dd.month_name, SUM(fs.line_total)
   FROM   northwind_dwh.fact_sales fs
   JOIN   northwind_dwh.dim_date dd ON dd.date_key = fs.order_date_key
   WHERE  dd.year = 1997
   GROUP BY dd.month_number, dd.month_name
   ORDER BY dd.month_number;
   ```
   Empaquetarla en función solo tiene sentido si se va a llamar desde muchos lugares.
</details>

### Ejercicio 12 — Función `info_pedido` con manejo de errores

Crea una función `info_pedido(p_order_id INTEGER)` que devuelva una `TABLE(campo TEXT, valor TEXT)` con los siguientes campos del pedido:

- `'cliente'` — `company_name`
- `'empleado'` — `first_name || ' ' || last_name`
- `'order_date'` — formateada como `'YYYY-MM-DD'`
- `'lineas'` — número de `order_details`
- `'total'` — `SUM(quantity * unit_price * (1 - discount))`

Si el pedido no existe, debe devolver una sola fila `('error', 'pedido no existe')`.

In [ ]:
%%sql
-- Tu solución aquí

<details>
<summary><strong>Solución</strong></summary>

```sql
CREATE OR REPLACE FUNCTION info_pedido(p_order_id INTEGER)
RETURNS TABLE(campo TEXT, valor TEXT) AS $$
DECLARE
    o RECORD;
BEGIN
    SELECT o.order_date, o.customer_id, o.employee_id,
           c.company_name,
           e.first_name || ' ' || e.last_name AS empleado,
           COUNT(od.product_id) AS lineas,
           SUM(od.quantity * od.unit_price * (1 - od.discount)) AS total
      INTO o
      FROM northwind_oltp.orders        o
      JOIN northwind_oltp.customers     c USING (customer_id)
      JOIN northwind_oltp.employees     e USING (employee_id)
      JOIN northwind_oltp.order_details od USING (order_id)
     WHERE o.order_id = p_order_id
     GROUP BY o.order_date, o.customer_id, o.employee_id,
              c.company_name, e.first_name, e.last_name;
    
    IF NOT FOUND THEN
        RETURN QUERY SELECT 'error'::TEXT, 'pedido no existe'::TEXT;
        RETURN;
    END IF;
    
    RETURN QUERY VALUES
        ('cliente',    o.company_name::TEXT),
        ('empleado',   o.empleado::TEXT),
        ('order_date', TO_CHAR(o.order_date, 'YYYY-MM-DD')),
        ('lineas',     o.lineas::TEXT),
        ('total',      ROUND(o.total, 2)::TEXT);
END;
$$ LANGUAGE plpgsql STABLE;

SELECT * FROM info_pedido(10248);
SELECT * FROM info_pedido(99999);
```

Patrón clave: `RETURN QUERY VALUES (...), (...), (...);` para devolver filas inline construidas a mano. Útil cuando quieres salida "key/value" en lugar de una tabla columnar.
</details>

## Limpieza

Si quieres eliminar las funciones y procedimientos que creaste durante los ejercicios:

In [ ]:
%%sql
DROP FUNCTION  IF EXISTS aplicar_iva(NUMERIC, NUMERIC);
DROP FUNCTION  IF EXISTS dias_envio(INTEGER);
DROP FUNCTION  IF EXISTS top_n_productos(INTEGER);
DROP FUNCTION  IF EXISTS tasa_de_envio_segura(INTEGER);
DROP PROCEDURE IF EXISTS reportar_top_clientes(INTEGER, NUMERIC);
DROP PROCEDURE IF EXISTS validar_dwh();
DROP FUNCTION  IF EXISTS info_pedido(INTEGER);

## Cierre del Tema 06

Lo que construiste a lo largo de los 5 notebooks:

| Notebook | Tema |
|---|---|
| **01** | Bloques anónimos `DO $$`, variables, `RAISE`, `SELECT INTO`, `PERFORM`, manejo de excepciones |
| **02** | Control de flujo: `IF`, `CASE`, `LOOP`, `FOR` (rango/query/array), `WHILE`, `CONTINUE`, etiquetas |
| **03** | Funciones: `CREATE FUNCTION`, escalares, `SETOF`, `TABLE`, `IMMUTABLE`/`STABLE`/`VOLATILE`, SQL vs plpgsql |
| **04** | Procedimientos: `CREATE PROCEDURE`, `CALL`, parámetros `IN`/`OUT`/`INOUT`, control de transacciones, cursores |
| **05** | Práctica integrada: 12 ejercicios graduales |

**Lo que sigue en el Tema 07:** funciones de ventana — `ROW_NUMBER`, `RANK`, `LAG`/`LEAD`, sumas acumulativas, promedios móviles. Aritmética por filas que ni `GROUP BY` ni PL/pgSQL hacen tan bien.

---

<p align="center">
<a href="04_procedimientos_y_cursores.ipynb">← Anterior: Notebook 04</a> | <a href="Readme.md">Volver al índice</a> | <a href="../Tema-07/Readme.md">Siguiente: Tema 07 →</a>
</p>